# 목표

연말정산 신고 안내 문서 활용 RAG 시스템 구현

In [1]:
# 기본 경로 설정
# ===============================================================
import os
import requests

PROJECT_NUM = "14"

ROOT_DIR = os.getcwd()


try:
    from google.colab import drive, userdata
    IS_COLAB_MODE = True
    print("코랩 모드")

except ModuleNotFoundError as e:
    IS_COLAB_MODE = False
    ROOT_DIR = os.path.abspath(os.path.join(ROOT_DIR, ".."))
    os.environ["KMP_DUPLICATE_LIB_OK"] = "True"
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    print(f"로컬 모드")


DATA_DIR = os.path.join(ROOT_DIR, "data")
RAW_DIR = os.path.join(DATA_DIR, "raw")
os.makedirs(DATA_DIR, exist_ok=True)


# 환경변수 로드 설정
def get_secret(key_name: str):
    if IS_COLAB_MODE:
        return userdata.get(key_name)
    else:
        from dotenv import load_dotenv
        load_dotenv(dotenv_path=os.path.join(ROOT_DIR, ".env"))
        return os.getenv(key_name)


if IS_COLAB_MODE:
    import subprocess

    drive.mount('/content/drive')

    # 압축 파일 확보
    if "raw.tar.gz" not in os.listdir():

        headers = {
            "Authorization": f"token {get_secret('GITHUB_PAT')}",
            "Accept": "application/vnd.github.v3+json"
        }

        release_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/tags/data-v{PROJECT_NUM}"

        response = requests.get(release_url, headers=headers)
        asset_id = response.json().get('assets', [])[0]["id"]

        download_url = f"https://api.github.com/repos/wonbywondev/ML-DL-data/releases/assets/{asset_id}"
        download_headers = headers.copy()
        download_headers["Accept"] = "application/octet-stream"

        with requests.get(download_url, headers=download_headers, stream=True) as r:
            r.raise_for_status()
            with open("raw.tar.gz", 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("· 압축 파일 있음")


    # 압축 해제 및 경로 처리
    if not os.path.exists(RAW_DIR):
        subprocess.run(["tar", "-xzvf", "raw.tar.gz", "-C", DATA_DIR], check=True)

    print("· 압축 해제 완료")
    print("· 환경 세팅 완료")


    DRIVE_DIR = os.path.join(ROOT_DIR, "drive", "MyDrive")
    SAVE_DIR = os.path.join(DRIVE_DIR, "runs", PROJECT_NUM)

    os.makedirs(SAVE_DIR, exist_ok=True)

PDF_PATH = os.path.join(RAW_DIR, "2024년+원천징수의무자를+위한+연말정산+신고안내.pdf")

로컬 모드


In [8]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

In [16]:
X = docs[2]
print(X.page_content)

국세청  
연말정산 서비스
이용자 서비스 내 용 접근 경로
공통
  원천징수(연말정산)안내 홈페이지 국세청 홈페이지(www.nts.go.kr) 
국세신고안내  개인 또는 법인  연말정산
  연말정산 관련 질의회신 및 판례 조회 국세법령정보시스템
(www.hometax.go.kr  법령정보)
  인터넷 상담 및 회신 국세청 국세상담센터
(www.hometax.go.kr  상담/제보)
근로자
소득공제
자료조회
  연말정산 소득·세액공제 자료 조회
홈택스  장려금·연말정산·기부금   
연말정산간소화
[ 안내 ]    126-내선1-5번
  현금영수증 발행금액 조회
현금영수증 
(홈택스  전자(세금)계산서·현금영수증· 
신용카드  현금영수증(근로자·소비자))
[ 안내 ]    126-내선1-1번
공제신고서
작성   간소화자료 선택 후 신고서 자동 반영
홈택스  장려금·연말정산·기부금   
편리한 연말정산
[ 안내 ]    126-내선1-5번
간편제출     작성된 공제신고서 및 증명자료  
온라인 제출
홈택스  장려금·연말정산·기부금   
편리한 연말정산
[ 안내 ]    126-내선1-5번
신고결과
조회
  과거 원천징수 영수증(지급명세서)조회
*   ’19년~’24년 및 ’25년 중 제출한  
’25년 귀속분(중도퇴사자 등)은 조회 가능
*   ’25.8월 수시오픈 이후부터는  
’19년 귀속 확인 불가
    제출된 연말정산 신고사항은 제출  
다음날부터 조회 가능
홈택스  My홈택스  연말정산   
지급명세서 등 제출내역
[ 안내 ]    126-내선1-3번
안내책자
및 영상
  「근로자를 위한 연말정산 안내」 책자
  「근로자를 위한 연말정산」 동영상
  「편리한 연말정산 이용방법」 동영상
국세청 홈페이지(www.nts.go.kr) 
국세신고안내  개인 또는 법인  연말정산
프로그램   연말정산 모의계산 서비스
국세청 홈택스(www.hometax.go.kr) 
장려금·연말정산·기부금  편리한  
연말정산  (모의계산) 

In [3]:
from img2table.document import PDF
from img2table.ocr import TesseractOCR
from img2table.document import Image


pdf = PDF(PDF_PATH, 
          pages=[2],
          detect_rotation=False,
          pdf_text_extraction=True)

ocr = TesseractOCR(n_threads=1, lang="eng")
doc = Image(PDF_PATH)

extracted_tables = pdf.extract_tables(
    ocr=ocr,
    implicit_rows=False,
    implicit_columns=False,
    borderless_tables=False,
    min_confidence=50
)

tesseract 5.5.2
 leptonica-1.87.0
  libgif 5.2.2 : libjpeg 8d (libjpeg-turbo 3.1.3) : libpng 1.6.54 : libtiff 4.7.1 : zlib 1.2.12 : libwebp 1.6.0 : libopenjp2 2.5.4
 Found NEON
 Found libarchive 3.8.5 zlib/1.2.12 liblzma/5.8.2 bz2lib/1.0.8 liblz4/1.10.0 libzstd/1.5.7 expat/expat_2.7.1 CommonCrypto/system libb2/system
 Found libcurl/8.7.1 SecureTransport (LibreSSL/3.3.6) zlib/1.2.12 nghttp2/1.67.1


In [25]:
for page_number, tables in extracted_tables.items():
    for idx, table in enumerate(tables):
        df = table.df
        print(f"Page {page_number} - Table {idx} DataFrame:")
        display(df) # 주피터 노트북 환경인 경우

Page 2 - Table 0 DataFrame:


,0,1,2,3
0,이용자,서비스,내 용,접근 경로
1,공통,원천징수(연말정산)안내 홈페이지,원천징수(연말정산)안내 홈페이지,국세청 홈페이지(www.nts.go.kr) \n국세신고안내  개인 또는 법인 ...
2,공통,연말정산 관련 질의회신 및 판례 조회,연말정산 관련 질의회신 및 판례 조회,국세법령정보시스템\n(www.hometax.go.kr  법령정보)
3,공통,인터넷 상담 및 회신,인터넷 상담 및 회신,국세청 국세상담센터\n(www.hometax.go.kr  상담/제보)
4,근로자,소득공제\n자료조회,연말정산 소득·세액공제 자료 조회,홈택스  장려금·연말정산·기부금 \n연말정산간소화\n[ 안내 ] 126-내선1-5번
5,근로자,소득공제\n자료조회,현금영수증 발행금액 조회,현금영수증\n(홈택스  전자(세금)계산서·현금영수증·\n신용카드  현금영수증(근...
6,근로자,공제신고서\n작성,간소화자료 선택 후 신고서 자동 반영,홈택스  장려금·연말정산·기부금 \n편리한 연말정산\n[ 안내 ] 126-내선1-5번
7,근로자,간편제출,작성된 공제신고서 및 증명자료\n온라인 제출,홈택스  장려금·연말정산·기부금 \n편리한 연말정산\n[ 안내 ] 126-내선1-5번
8,근로자,신고결과\n조회,과거 원천징수 영수증(지급명세서)조회\n* ’19년~’24년 및 ’25년 중 제출한...,홈택스  My홈택스  연말정산 \n지급명세서 등 제출내역\n[ 안내 ] 126...
9,근로자,안내책자\n및 영상,「근로자를 위한 연말정산 안내」 책자\n「근로자를 위한 연말정산」 동영상\n「편리한...,국세청 홈페이지(www.nts.go.kr) \n국세신고안내  개인 또는 법인 ...


In [ ]:
data = extracted_tables[2][0].content.values()
data

odict_values([[TableCell(bbox=BBox(x1=173, y1=354, x2=292, y2=409), value='이용자'), TableCell(bbox=BBox(x1=292, y1=354, x2=425, y2=409), value='서비스'), TableCell(bbox=BBox(x1=425, y1=354, x2=867, y2=409), value='내 용'), TableCell(bbox=BBox(x1=867, y1=354, x2=1323, y2=409), value='접근 경로')], [TableCell(bbox=BBox(x1=173, y1=409, x2=292, y2=622), value='공통'), TableCell(bbox=BBox(x1=292, y1=409, x2=867, y2=480), value='원천징수(연말정산)안내 홈페이지'), TableCell(bbox=BBox(x1=292, y1=409, x2=867, y2=480), value='원천징수(연말정산)안내 홈페이지'), TableCell(bbox=BBox(x1=867, y1=409, x2=1323, y2=480), value='국세청 홈페이지(www.nts.go.kr) \ue3fc\n국세신고안내 \ue3fc 개인 또는 법인 \ue3fc 연말정산')], [TableCell(bbox=BBox(x1=173, y1=409, x2=292, y2=622), value='공통'), TableCell(bbox=BBox(x1=292, y1=480, x2=867, y2=551), value='연말정산 관련 질의회신 및 판례 조회'), TableCell(bbox=BBox(x1=292, y1=480, x2=867, y2=551), value='연말정산 관련 질의회신 및 판례 조회'), TableCell(bbox=BBox(x1=867, y1=480, x2=1323, y2=551), value='국세법령정보시스템\n(www.hometax.go.kr \ue3fc 법령정보)')], [TableCel

In [19]:
import pandas as pd

a = pd.DataFrame(data)

In [20]:
a

,0,1,2,3
0,"TableCell(bbox=BBox(x1=173, y1=354, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=354, x2=425, y2...","TableCell(bbox=BBox(x1=425, y1=354, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=354, x2=1323, y..."
1,"TableCell(bbox=BBox(x1=173, y1=409, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=409, x2=867, y2...","TableCell(bbox=BBox(x1=292, y1=409, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=409, x2=1323, y..."
2,"TableCell(bbox=BBox(x1=173, y1=409, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=480, x2=867, y2...","TableCell(bbox=BBox(x1=292, y1=480, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=480, x2=1323, y..."
3,"TableCell(bbox=BBox(x1=173, y1=409, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=551, x2=867, y2...","TableCell(bbox=BBox(x1=292, y1=551, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=551, x2=1323, y..."
4,"TableCell(bbox=BBox(x1=173, y1=622, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=622, x2=425, y2...","TableCell(bbox=BBox(x1=425, y1=622, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=622, x2=1323, y..."
5,"TableCell(bbox=BBox(x1=173, y1=622, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=622, x2=425, y2...","TableCell(bbox=BBox(x1=425, y1=725, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=725, x2=1323, y..."
6,"TableCell(bbox=BBox(x1=173, y1=622, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=859, x2=425, y2...","TableCell(bbox=BBox(x1=425, y1=859, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=859, x2=1323, y..."
7,"TableCell(bbox=BBox(x1=173, y1=622, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=957, x2=425, y2...","TableCell(bbox=BBox(x1=425, y1=957, x2=867, y2...","TableCell(bbox=BBox(x1=867, y1=957, x2=1323, y..."
8,"TableCell(bbox=BBox(x1=173, y1=622, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=1055, x2=425, y...","TableCell(bbox=BBox(x1=425, y1=1055, x2=867, y...","TableCell(bbox=BBox(x1=867, y1=1055, x2=1323, ..."
9,"TableCell(bbox=BBox(x1=173, y1=622, x2=292, y2...","TableCell(bbox=BBox(x1=292, y1=1291, x2=425, y...","TableCell(bbox=BBox(x1=425, y1=1291, x2=867, y...","TableCell(bbox=BBox(x1=867, y1=1291, x2=1323, ..."


In [24]:
import pdfplumber

pdf = pdfplumber.open(PDF_PATH)
X = pdf.pages[2]
im = X.to_image()
text = X.extract_text(keep_black_chars=True)

In [26]:
print(text)

국세청
연말정산 서비스
이용자 서비스 내 용 접근 경로
국세청 홈페이지(www.nts.go.kr) 
원천징수(연말정산)안내 홈페이지
국세신고안내  개인 또는 법인  연말정산
국세법령정보시스템
공통 연말정산 관련 질의회신 및 판례 조회
(www.hometax.go.kr  법령정보)
국세청 국세상담센터
인터넷 상담 및 회신
(www.hometax.go.kr  상담/제보)
홈택스  장려금·연말정산·기부금 
연말정산 소득·세액공제 자료 조회 연말정산간소화
[ 안내 ] 126-내선1-5번
소득공제
자료조회 현금영수증
(홈택스  전자(세금)계산서·현금영수증·
현금영수증 발행금액 조회
신용카드  현금영수증(근로자·소비자))
[ 안내 ] 126-내선1-1번
홈택스  장려금·연말정산·기부금 
공제신고서
간소화자료 선택 후 신고서 자동 반영 편리한 연말정산
작성
[ 안내 ] 126-내선1-5번
홈택스  장려금·연말정산·기부금 
작 성된 공제신고서 및 증명자료
간편제출 편리한 연말정산
온라인 제출
[ 안내 ] 126-내선1-5번
근로자
과거 원천징수 영수증(지급명세서)조회
* ’ 19년~’24년 및 ’25년 중 제출한
’25년 귀속분(중도퇴사자 등)은 조회 가능 홈택스  My홈택스  연말정산 
신고결과
* ’ 25.8월 수시오픈 이후부터는 지급명세서 등 제출내역
조회
’19년 귀속 확인 불가 [ 안내 ] 126-내선1-3번
제 출된 연말정산 신고사항은 제출
다음날부터 조회 가능
「근로자를 위한 연말정산 안내」 책자
안내책자 국세청 홈페이지(www.nts.go.kr) 
「근로자를 위한 연말정산」 동영상
및 영상 국세신고안내  개인 또는 법인  연말정산
「편리한 연말정산 이용방법」 동영상
국세청 홈택스(www.hometax.go.kr) 
프로그램 연말정산 모의계산 서비스 장려금·연말정산·기부금  편리한
연말정산  (모의계산) 연말정산 자동계산
원천징수이행상황신고 및 환급신청
원천세 전자신고 설명서
지급명세서 제출 홈택스(www.hometax.g

In [ ]:

# TABLE_SETTINGS = {
#     "vertical_strategy": "lines",
#     "horizontal_strategy": "lines",
#     "explicit_vertical_lines": [],
#     "explicit_horizontal_lines": [],
#     "snap_tolerance": 3,
#     "snap_x_tolerance": 2,
#     "snap_y_tolerance": 2,
#     "join_tolerance": 3,
#     "join_x_tolerance": 3,
#     "join_y_tolerance": 3,
#     "edge_min_length": 3,
#     "min_words_vertical": 3,
#     "min_words_horizontal": 1,
#     "intersection_tolerance": 3,
#     "intersection_x_tolerance": 3,
#     "intersection_y_tolerance": 3,
#     "text_tolerance": 3,
#     "text_x_tolerance": 3,
#     "text_y_tolerance": 3,
# }

all_pages = []

with pdfplumber.open(PDF_PATH) as pdf:
    pages = pdf.pages

    for page in pages:

        tables = page.extract_tables()

        all_pages.extend(tables)

In [7]:
len(all_pages)

250

In [ ]:
import random

for i in range(10):

    print(docs[i].page_content)
    print("="*30)

연말정산
신고안내
일 하나는 제대로 하는,
국민께 인정받는 국세청
2024. 12.
2024 원천징수의무자를 위한
맞춤형 안내
간소화 서비스
 일괄제공 서비스
발간등록번호
11-1210000-000072-10
머 리 말
어려운 경제 상황 속에서도 항상 성실하게 원천징수의무를 이행하여 국세행정에  
협력해주신 원천징수의무자(회사)와 성실하게 세금을 납부하신 근로자 여러분께 진심
으로 감사드립니다.
그동안 국세청은 「연말정산 미리보기」, 「편리한 연말정산」 시스템과 「간소화자료  
일괄제공」 서비스를 시행하는 등 디지털 납세서비스를 제공하고 「맞춤형안내」를 통해 
성실 납세를 지원해 왔습니다.
더불어, 2024년 귀속부터는 소득기준 초과 부양가족에 대한 자료 조회 및 다운로드를  
제한하여 납세자의 실수를 최소화하고 정확한 연말정산을 지원하기 위해 「간소화  
서비스」를 전면 개편합니다.
이 책자에는 세법 개정사항과 최신 예규, 소득·세액공제신고서 작성 방법, 홈택스를 
이용한 지급명세서 제출 방법 등을 분야별로 담아 납세자에게 실질적인 도움이 될 수 
있도록 구성하였습니다.
책자에 수록된 내용 외에도 연말정산 주요정보, 동영상 자료, 계산 사례, 체크리스트 등 
다양한 자료를 국세청 누리집(www.nts.go.kr) 연말정산 종합안내에서 제공하고 있습니다.
본 책자가 연말정산에 관한 납세자 여러분의 궁금증을 해소하고, 원천징수의무 이행에 
유용한 참고자료로 활용되어 성실신고·납부에 도움이 되기를 바랍니다.
2024년 12월
법인납세국장     
연말정산
신고안내
2024 원천징수의무자를 위한
국세청  
연말정산 서비스
이용자 서비스 내 용 접근 경로
공통
  원천징수(연말정산)안내 홈페이지 국세청 홈페이지(www.nts.go.kr) 
국세신고안내  개인 또는 법인  연말정산
  연말정산 관련 질의회신 및 판례 조회 국세법령정보시스템
(www.hometax.go.kr  법령정보)
  인터넷 상담 및 회신 국세청 국세상담센터
(www.hometax.go.kr